# CHATBOT WITH LANGCHAIN FAISSDB OPENAI LLM

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter as RCTS
from langchain_huggingface import HuggingFaceEmbeddings
import os
from langchain_community.vectorstores import FAISS
from openai import OpenAI

C:\Users\KOUSHIK\AppData\Local\Temp\ipykernel_20468\751780106.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\KOUSHIK\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Load PDF

In [3]:
loader = PyPDFLoader('tcs.pdf')

In [4]:
doc = loader.load()

In [5]:
print(type(doc))
print("Number of pages:", len(doc))
print(doc[0])

<class 'list'>
Number of pages: 360
page_content='Integrated
/A.caltnnual Report
2025-26
Leadership in 
Enterprise /A.caltI
Infrastructure 
to Intelligence' metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.3 (Windows)', 'creationdate': '2026-05-15T23:33:43+05:30', 'moddate': '2026-08-12T17:07:19+05:30', 'title': 'TCS Annual Report 2025-26', 'trapped': '/false', 'source': 'tcs.pdf', 'total_pages': 360, 'page': 0, 'page_label': '1'}


In [6]:
print(doc[0].page_content)

Integrated
/A.caltnnual Report
2025-26
Leadership in 
Enterprise /A.caltI
Infrastructure 
to Intelligence


In [7]:
print(doc[0].metadata)

{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.3 (Windows)', 'creationdate': '2026-05-15T23:33:43+05:30', 'moddate': '2026-08-12T17:07:19+05:30', 'title': 'TCS Annual Report 2025-26', 'trapped': '/false', 'source': 'tcs.pdf', 'total_pages': 360, 'page': 0, 'page_label': '1'}


In [8]:
print(type(doc[0]))

<class 'langchain_core.documents.base.Document'>


In [9]:
# 2. Split PDF into chunks

In [10]:
chunking = RCTS(chunk_size = 2500, chunk_overlap = 500)

In [11]:
chunks = chunking.split_documents(doc)

In [12]:
print(type(chunks))
print("Number of chunks:", len(chunks))
print(chunks[0])

<class 'list'>
Number of chunks: 640
page_content='Integrated
/A.caltnnual Report
2025-26
Leadership in 
Enterprise /A.caltI
Infrastructure 
to Intelligence' metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.3 (Windows)', 'creationdate': '2026-05-15T23:33:43+05:30', 'moddate': '2026-08-12T17:07:19+05:30', 'title': 'TCS Annual Report 2025-26', 'trapped': '/false', 'source': 'tcs.pdf', 'total_pages': 360, 'page': 0, 'page_label': '1'}


In [13]:
print(type(chunks[0]))

<class 'langchain_core.documents.base.Document'>


In [14]:
# 3. Create embedding model

In [15]:

model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 103.28it/s]


In [16]:
# 4. create faiss db and pass documents and vector model

In [17]:
vector_db = FAISS.from_documents(chunks, model)

In [18]:
print(type(vector_db))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [19]:
# 5. make a prompt

In [20]:
question = "What was the net profit reported?"

In [21]:
context = vector_db.similarity_search(question, k = 3)

In [22]:
results = ''
for i in context:
    results += i.page_content+'\n\n'

In [23]:
# print(results)

In [24]:
prompt = f"""
		Answer only from the <context> below. Do not use any prior knowledge. 
        If the answer is absent, reply exactly: 'I could not find the answer in the provided context.'
        Do not guess. Cite the supporting sentence.
        
        Context: {results}. 
        Question: {question}.
	"""

In [25]:
# 6. give prompt to open ai llm and get response

In [26]:
client = OpenAI(
    api_key="AQ.Ab8RN6L0TImjkvagvnPSg17I2_IQq1pOw6WlSVNL8aTajIai7Q",   
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [27]:
response = client.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=[{"role": "user", "content": prompt}]
)

In [30]:
for i in response.choices:
    print(i.message.content)
    print("-" * 40)

Based on the provided context, the profit for the year (net profit) was 49,454 (for the year ended March 31, 2026) and 48,797 (for the year ended March 31, 2025).

Supporting sentences:
* "Profit for the year 49,454 48,797"
----------------------------------------
